# 🎓 NLP Final Project – Phân Loại Phản Hồi Sinh Viên
### Student Feedback Analysis with Deep Learning

**Bài toán:** Text Classification 4 nhãn · **Ngôn ngữ:** Tiếng Việt  
**5 Mô hình:** KimCNN · BiLSTM+Attention · RCNN · Transformer · PhoBERT

---

In [1]:
!pip install transformers sentencepiece -q

import os, re, csv, math, time, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
from collections import Counter, defaultdict
from IPython.display import display, Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
    confusion_matrix, roc_auc_score)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
os.makedirs('/content/outputs', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f' Import xong!')
print(f'  Device: {DEVICE}' + (f'  —  {torch.cuda.get_device_name(0)}' if DEVICE=='cuda' else ''))


 Import xong!
  Device: cuda  —  Tesla T4


## ⚙️ Phần 1 — Cấu Hình (Config)

In [2]:

# ── Nhãn ───────────────────────────────────────────────
LABEL2ID = {'Tích cực': 0, 'Tiêu cực': 1, 'Góp ý': 2, 'Trung lập': 3}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 4

# ── Chia tập ────────────────────────────────────────────
RANDOM_SEED = 42

# ── Tokenization ────────────────────────────────────────
MAX_SEQ_LEN = 128
VOCAB_SIZE   = 10_000
EMBED_DIM    = 128

# ── Huấn luyện ──────────────────────────────────────────
BATCH_SIZE    = 32
LR            = 1e-3
BERT_LR       = 2e-5
EPOCHS        = 10
PATIENCE      = 3
WEIGHT_DECAY  = 1e-4
SEEDS         = [42, 123, 777]

# ── Kiến trúc ───────────────────────────────────────────
CNN_FILTER_SIZES = [2, 3, 4]
CNN_FILTERS      = 128
CNN_DROP         = 0.5

LSTM_HIDDEN  = 128
LSTM_LAYERS  = 2
LSTM_DROP    = 0.3

TRANS_D      = 128
TRANS_HEADS  = 4
TRANS_LAYERS = 2
TRANS_FF     = 256
TRANS_DROP   = 0.1

BERT_MODEL   = 'vinai/phobert-base'
BERT_HIDDEN  = 768
BERT_DROP    = 0.1

def set_seed(s=RANDOM_SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed()
print(' Config OK')
print(f'   NUM_CLASSES={NUM_CLASSES}  BATCH={BATCH_SIZE}  EPOCHS={EPOCHS}  SEEDS={SEEDS}')


 Config OK
   NUM_CLASSES=4  BATCH=32  EPOCHS=10  SEEDS=[42, 123, 777]


## 📂 Phần 2 — Dữ Liệu

In [3]:
set_seed()

# ─── Templates ─────────────────────────────────────────
POS_TPL = [
    "Thầy/Cô dạy rất {ap}, tôi học được {n} rất nhiều.",
    "Môn học này {ap} hơn tôi nghĩ, nội dung thiết thực và cập nhật.",
    "Tôi rất {av} với cách giảng dạy, bài giảng {ap} và dễ hiểu.",
    "Khóa học {ap} thực sự, tôi học được nhiều kiến thức {n} bổ ích.",
    "Thầy/Cô giảng rất {ap}, luôn {vp} sinh viên trong quá trình học.",
    "Tài liệu học tập {ap} và đầy đủ, giúp tôi hiểu bài nhanh hơn.",
    "Phương pháp giảng dạy rất {ap}, kết hợp tốt lý thuyết với thực hành.",
    "Tôi {av} với môn này, nó giúp tôi phát triển kỹ năng {n} đáng kể.",
    "Bài tập và dự án trong môn rất {ap} và gắn liền với thực tế.",
    "Thầy/Cô luôn {vp} và giải đáp thắc mắc tận tình.",
    "Nội dung môn học được cập nhật {ap}, phù hợp với xu hướng hiện tại.",
    "Môn học cung cấp nền tảng {ap} cho công việc sau này của tôi.",
    "Giờ học rất {ap} và sôi nổi nhờ các hoạt động tương tác.",
    "Cách đánh giá trong môn {ap} và công bằng với tất cả sinh viên.",
    "Tôi rất thích cách thầy/cô {vp} ví dụ thực tế vào bài giảng.",
    "Học phần này giúp tôi {vp} tư duy phân tích và giải quyết vấn đề.",
    "Bài giảng {ap}, dễ theo dõi và dễ ghi nhớ.",
    "Thầy/Cô tạo môi trường học tập {ap}, khuyến khích đặt câu hỏi.",
    "Tôi học được nhiều kỹ năng {n} quan trọng từ môn học này.",
    "Chương trình học được thiết kế {ap}, logic và hệ thống.",
    "Tôi đặc biệt ấn tượng với các buổi thảo luận {ap}.",
    "Môn học này xứng đáng được đánh giá {ap} về mọi mặt.",
    "Thầy/Cô rất {ap} trong việc hỗ trợ sinh viên ngoài giờ.",
    "Kiến thức từ môn này rất {ap} và áp dụng được ngay thực tế.",
    "Nhìn chung, tôi rất {av} với toàn bộ môn học này.",
]
POS_F = {
    'ap': ['tốt','xuất sắc','tuyệt vời','hiệu quả','chuyên nghiệp','hay','thú vị','ấn tượng','chất lượng','bổ ích'],
    'av': ['hài lòng','ấn tượng','thích thú','vui','biết ơn'],
    'n':  ['chuyên môn','kỹ năng','kiến thức','kinh nghiệm','tư duy'],
    'vp': ['hỗ trợ','khuyến khích','truyền cảm hứng','lôi cuốn','hướng dẫn','phát triển','rèn luyện'],
}

NEG_TPL = [
    "Thầy/Cô giảng rất {an}, tôi không thể hiểu được nội dung.",
    "Môn học này {an} và không thực tế, nội dung hoàn toàn lạc hậu.",
    "Tôi {av} với cách tổ chức môn học này, rất {an} và thiếu chuyên nghiệp.",
    "Tài liệu học tập {an} và thiếu cập nhật, gây nhiều khó khăn.",
    "Thầy/Cô hay đến lớp muộn, ảnh hưởng nghiêm trọng đến giờ học.",
    "Phương pháp giảng dạy {an}, chỉ đọc slide mà không giải thích gì thêm.",
    "Bài tập quá {an} và không có hướng dẫn rõ ràng cho sinh viên.",
    "Thầy/Cô không trả lời câu hỏi của sinh viên, gây rất nản lòng.",
    "Môn học thiếu thực tiễn, quá nặng lý thuyết {an}.",
    "Lịch học {an} và thường bị thay đổi đột ngột không báo trước.",
    "Tôi cảm thấy {av} vì không học được gì có ích từ môn này.",
    "Nội dung kiểm tra {an} so với những gì được dạy trên lớp.",
    "Cơ sở vật chất phục vụ môn này {an}, không đáp ứng nhu cầu học tập.",
    "Thầy/Cô giảng quá nhanh, sinh viên không thể theo kịp.",
    "Môn học này {an} về mọi mặt, cần được cải thiện toàn diện.",
    "Điểm số không phản ánh đúng năng lực, cách chấm điểm {an}.",
    "Sinh viên không nhận được phản hồi kịp thời sau khi nộp bài.",
    "Thầy/Cô có thái độ {an} với sinh viên khi họ đặt câu hỏi.",
    "Bài kiểm tra không phù hợp nội dung đã học, rất {an}.",
    "Tôi thấy đây là môn {an} nhất trong học kỳ.",
]
NEG_F = {
    'an': ['kém','chán','nhàm','tệ','không hay','khó hiểu','lạc hậu','thiếu sót','không ổn','yếu'],
    'av': ['thất vọng','không hài lòng','bực bội','chán nản','thất vọng nặng nề'],
}

SUG_TPL = [
    "Tôi đề nghị nên {vs} thêm các {n} thực tế vào bài giảng.",
    "Môn học nên {vs} thêm {n} để sinh viên áp dụng kiến thức tốt hơn.",
    "Theo tôi, cần {vs} phương pháp giảng dạy để {as_} hơn cho sinh viên.",
    "Thầy/Cô nên {vs} thêm tài liệu tham khảo {as_} cho sinh viên.",
    "Tôi nghĩ nên có thêm {n} thực hành để hiểu rõ lý thuyết hơn.",
    "Nếu {vs} hơn, môn học sẽ {as_} hơn rất nhiều.",
    "Đề xuất {vs} cách đánh giá, bổ sung thêm các hình thức {n} khác.",
    "Cần {vs} lịch học cho hợp lý hơn, tránh {n} quá dày đặc.",
    "Thầy/Cô nên {vs} phản hồi sớm hơn để sinh viên cải thiện bài làm.",
    "Tôi muốn kiến nghị {vs} thêm giờ thảo luận nhóm trong môn học.",
    "Nên {vs} các công cụ học tập hiện đại vào quá trình giảng dạy.",
    "Đề nghị {vs} tốc độ giảng bài để sinh viên dễ theo dõi hơn.",
    "Môn học cần {vs} nội dung cho phù hợp với thực tiễn công việc hiện nay.",
    "Tôi đề xuất {vs} nhiều case study hơn để tăng tính ứng dụng.",
    "Nên {vs} bài kiểm tra theo hướng đánh giá kỹ năng thực hành.",
    "Thầy/Cô có thể {vs} thêm video minh họa để bài giảng {as_} hơn.",
    "Kiến nghị {vs} tài liệu học tập, bổ sung các bài đọc {as_} hơn.",
    "Nên tổ chức thêm các buổi {n} với chuyên gia trong ngành.",
    "Đề nghị {vs} cơ chế hỏi đáp để sinh viên dễ tiếp cận hơn.",
    "Cần {vs} sự cân bằng giữa lý thuyết và thực hành trong môn.",
    "Nên {vs} thêm bài tập nhóm để phát triển kỹ năng làm việc nhóm.",
    "Tôi nghĩ nên {vs} thêm môi trường thực hành lab cho sinh viên.",
]
SUG_F = {
    'vs':  ['bổ sung','cập nhật','điều chỉnh','cải thiện','tăng cường','đưa vào','giảm bớt','tổ chức','sắp xếp lại'],
    'n':   ['ví dụ','bài tập','dự án','buổi học','nội dung','tài liệu','bài kiểm tra','lịch học','cuộc giao lưu'],
    'as_': ['hấp dẫn','thực tế','hiệu quả','dễ hiểu','phong phú','đa dạng','sinh động'],
}

NEU_TPL = [
    "Môn {n} gồm {num} tín chỉ, học vào {t}.",
    "Thầy/Cô đã trình bày {num} chương trong học kỳ này.",
    "Môn học sử dụng sách giáo trình {n} làm tài liệu chính.",
    "Lớp học có khoảng {num} sinh viên theo học môn này.",
    "Bài kiểm tra giữa kỳ diễn ra vào tuần thứ {num} của học kỳ.",
    "Điểm cuối kỳ chiếm {num}% tổng điểm của môn học.",
    "Môn học yêu cầu sinh viên hoàn thành {num} bài tập lớn.",
    "Tài liệu được đăng tải trên hệ thống {n} của trường.",
    "Các buổi học được tổ chức {num} lần mỗi tuần vào {t}.",
    "Kết quả học tập được đánh giá qua {num} bài kiểm tra.",
    "Môn học này thuộc khối {n} trong chương trình đào tạo.",
    "Thời lượng mỗi buổi học là {num} tiết.",
    "Sinh viên cần đạt điểm tối thiểu {num} để qua môn.",
    "Môn học có tổng cộng {num} buổi học trong học kỳ.",
    "Giảng viên phụ trách thuộc khoa {n}.",
    "Sinh viên được phép vắng tối đa {num} buổi trong học kỳ.",
    "Lớp diễn ra tại phòng {n} vào mỗi {t}.",
    "Đề thi gồm {num} câu trắc nghiệm và {num} câu tự luận.",
    "Môn học này có {num} tín chỉ lý thuyết và {num} tín chỉ thực hành.",
    "Thời gian làm bài kiểm tra là {num} phút.",
]
NEU_F = {
    'n':   ['học phần','giáo trình','LMS','kiến thức chuyên ngành','tự chọn','bắt buộc','Công nghệ thông tin','Kinh tế','tập trung'],
    'num': ['2','3','4','5','6','7','8','10','12','15','30','40','50','60','70','80','90'],
    't':   ['thứ Hai','thứ Ba','thứ Tư','thứ Năm','thứ Sáu','buổi sáng','buổi chiều'],
}

def fill(template, fills):
    result = template
    for key, opts in fills.items():
        ph = '{' + key + '}'
        while ph in result:
            result = result.replace(ph, random.choice(opts), 1)
    return result

def make_samples(tmpls, fills, n, label_id, label_name):
    prefixes = ['','','','Theo tôi, ','Thật ra, ','Nhìn chung, ','Cá nhân tôi thấy ']
    rows = []
    for _ in range(n):
        text = fill(random.choice(tmpls), fills)
        if random.random() < 0.15: text = text.lower()
        if random.random() < 0.2 and text.endswith('.'): text = text[:-1] + '!'
        text = (random.choice(prefixes) + text).strip()
        rows.append({'text': text, 'label': label_id, 'label_name': label_name})
    return rows

all_rows = (
    make_samples(POS_TPL, POS_F, 530, 0, 'Tích cực') +
    make_samples(NEG_TPL, NEG_F, 520, 1, 'Tiêu cực') +
    make_samples(SUG_TPL, SUG_F, 500, 2, 'Góp ý')    +
    make_samples(NEU_TPL, NEU_F, 500, 3, 'Trung lập')
)
random.shuffle(all_rows)

df = pd.DataFrame(all_rows)
DATA_PATH = '/content/student_feedback.csv'
df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

print(f'Dataset: {DATA_PATH}')
print(f'Tổng {len(df)} mẫu:')
for name, cnt in df['label_name'].value_counts().items():
    print(f'   {name}: {cnt}')


Dataset: /content/student_feedback.csv
Tổng 2050 mẫu:
   Tích cực: 530
   Tiêu cực: 520
   Trung lập: 500
   Góp ý: 500


In [4]:
df['n_words'] = df['text'].str.split().str.len()

print('='*55)
print('THỐNG KÊ BỘ DỮ LIỆU')
print('='*55)
print(f'Tổng mẫu        : {len(df)}')
print(f'Độ dài TB (từ)  : {df.n_words.mean():.1f}')
print(f'Độ dài median   : {df.n_words.median():.1f}')
print(f'Min / Max       : {df.n_words.min()} / {df.n_words.max()}')
print()
counts = df['label_name'].value_counts()
for name, cnt in counts.items():
    print(f'  {name:<12}: {cnt:>5} ({cnt/len(df)*100:.1f}%)')
print('='*55)

# ── Biểu đồ EDA ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
COLORS = ['#4CAF50','#F44336','#FF9800','#2196F3']

# 1. Phân phối nhãn
axes[0].bar(counts.index, counts.values, color=COLORS, edgecolor='white', lw=1.5)
axes[0].set_title('Phân phối nhãn', fontweight='bold')
axes[0].set_ylabel('Số mẫu')
for i,(_, v) in enumerate(counts.items()):
    axes[0].text(i, v+3, str(v), ha='center', fontweight='bold')
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# 2. Phân phối độ dài
for i, (name, grp) in enumerate(df.groupby('label_name')):
    axes[1].hist(grp['n_words'], bins=20, alpha=0.65, label=name, color=COLORS[i])
axes[1].set_title('Độ dài văn bản theo nhãn', fontweight='bold')
axes[1].set_xlabel('Số từ'); axes[1].set_ylabel('Tần suất')
axes[1].legend(fontsize=8); axes[1].grid(axis='y', linestyle='--', alpha=0.4)

# 3. Boxplot độ dài
data_box = [df[df['label_name']==n]['n_words'].values for n in counts.index]
bp = axes[2].boxplot(data_box, labels=counts.index, patch_artist=True)
for patch, c in zip(bp['boxes'], COLORS):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[2].set_title('Boxplot độ dài theo nhãn', fontweight='bold')
axes[2].set_ylabel('Số từ'); axes[2].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('/content/outputs/eda.png', dpi=150, bbox_inches='tight')
plt.show()


THỐNG KÊ BỘ DỮ LIỆU
Tổng mẫu        : 2050
Độ dài TB (từ)  : 14.9
Độ dài median   : 15.0
Min / Max       : 8 / 22

  Tích cực    :   530 (25.9%)
  Tiêu cực    :   520 (25.4%)
  Trung lập   :   500 (24.4%)
  Góp ý       :   500 (24.4%)


## 🔧 Phần 3 — Tiền Xử Lý & DataLoader

In [5]:

# ── 1. Chuẩn hoá văn bản ───────────────────────────────
def normalize_text(text: str) -> str:
    text = unicodedata.normalize('NFC', text).lower()
    text = re.sub(r'[^\w\s,\.!\?]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

# ── 2. Vocabulary ──────────────────────────────────────
class Vocabulary:
    PAD, UNK = '<PAD>', '<UNK>'
    def __init__(self, max_size=VOCAB_SIZE):
        self.max_size = max_size
        self.token2id = {self.PAD: 0, self.UNK: 1}
        self.id2token = {0: self.PAD, 1: self.UNK}

    def build(self, texts, min_freq=1):
        ctr = Counter()
        for t in texts:
            ctr.update(normalize_text(t).split())
        for tok, freq in ctr.most_common(self.max_size - 2):
            if freq >= min_freq:
                idx = len(self.token2id)
                self.token2id[tok] = idx
                self.id2token[idx] = tok
        print(f'   Vocabulary: {len(self.token2id):,} từ  |  OOV sẽ map → UNK')
        return self

    def encode(self, text, max_len=MAX_SEQ_LEN):
        ids = [self.token2id.get(t, 1) for t in normalize_text(text).split()[:max_len]]
        return ids + [0] * (max_len - len(ids))   # pad to max_len

    def __len__(self): return len(self.token2id)

# ── 3. Dataset ─────────────────────────────────────────
class FeedbackDataset(Dataset):
    def __init__(self, texts, labels, vocab=None, tokenizer=None, max_len=MAX_SEQ_LEN):
        self.texts, self.labels = texts, labels
        self.vocab, self.tokenizer, self.max_len = vocab, tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text  = self.texts[idx]
        label = self.labels[idx]
        if self.tokenizer is not None:
            enc = self.tokenizer(
                normalize_text(text), max_length=self.max_len,
                padding='max_length', truncation=True, return_tensors='pt')
            return {'input_ids':      enc['input_ids'].squeeze(0),
                    'attention_mask': enc['attention_mask'].squeeze(0),
                    'label':          torch.tensor(label, dtype=torch.long)}
        ids = self.vocab.encode(text, self.max_len)
        return {'input_ids': torch.tensor(ids, dtype=torch.long),
                'label':     torch.tensor(label, dtype=torch.long)}

# ── 4. Chia tập 70 / 10 / 20 ──────────────────────────
texts_all  = df['text'].tolist()
labels_all = df['label'].tolist()

# Bước 1: tách 70% train vs 30% còn lại
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    texts_all, labels_all,
    test_size=0.30, random_state=RANDOM_SEED, stratify=labels_all)

# Bước 2: chia 30% còn lại → val(10%) + test(20%)
# 10 / (10+20) = 0.333  →  test_size = 0.667
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp,
    test_size=0.667, random_state=RANDOM_SEED, stratify=y_tmp)

print(f'📊 Chia tập (stratified 70/10/20):')
print(f'   Train : {len(X_tr):>5}  |  Val : {len(X_val):>5}  |  Test : {len(X_te):>5}')
print(f'   Tổng  : {len(X_tr)+len(X_val)+len(X_te)}')

# ── 5. Vocabulary ──────────────────────────────────────
print()
vocab = Vocabulary().build(X_tr)

# ── 6. DataLoaders ─────────────────────────────────────
def make_loaders(use_vocab=True, tokenizer=None, bs=BATCH_SIZE):
    v = vocab if use_vocab else None
    tr = DataLoader(FeedbackDataset(X_tr,  y_tr,  v, tokenizer), bs, shuffle=True,  drop_last=False)
    va = DataLoader(FeedbackDataset(X_val, y_val, v, tokenizer), bs, shuffle=False)
    te = DataLoader(FeedbackDataset(X_te,  y_te,  v, tokenizer), bs, shuffle=False)
    return tr, va, te

train_loader, val_loader, test_loader = make_loaders()

# ── 7. OOV rate ────────────────────────────────────────
all_test_toks = [t for sent in X_te for t in normalize_text(sent).split()]
oov = sum(1 for t in all_test_toks if t not in vocab.token2id)
print(f'   OOV rate (test): {oov/len(all_test_toks)*100:.1f}%')
print()


📊 Chia tập (stratified 70/10/20):
   Train :  1435  |  Val :   204  |  Test :   411
   Tổng  : 2050

   Vocabulary: 559 từ  |  OOV sẽ map → UNK
   OOV rate (test): 0.1%



## 🤖 Phần 4 — Định Nghĩa 5 Mô Hình

In [6]:
class KimCNN(nn.Module):
    """
    CNN cho phân loại văn bản — Kim (2014), ACL.
    Kiến trúc:
        Embedding(V, E)
        → Conv1D (k=2), Conv1D (k=3), Conv1D (k=4)  [mỗi kernel: F filters]
        → ReLU + MaxPool-over-time
        → Concat  →  Dropout  →  Linear(3F, C)
    """
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                 filter_sizes=CNN_FILTER_SIZES, num_filters=CNN_FILTERS,
                 num_classes=NUM_CLASSES, dropout=CNN_DROP):
        super().__init__()
        self.name = 'KimCNN'
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k) for k in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, input_ids, **kw):
        # (B, L) → (B, E, L)
        x = self.embedding(input_ids).permute(0, 2, 1)
        # Conv → ReLU → MaxPool → (B, F)
        pooled = []
        for conv in self.convs:
            c = F.relu(conv(x))                    # (B, F, L-k+1)
            p = F.max_pool1d(c, c.size(2)).squeeze(2)  # (B, F)
            pooled.append(p)
        out = self.dropout(torch.cat(pooled, dim=1))   # (B, 3F)
        return self.fc(out)                            # (B, C)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

n_params = KimCNN().count_params()
print(f'   KimCNN định nghĩa xong')
print(f'   Số tham số ≈ {n_params:,}')
print(f'   Kiến trúc : Emb({VOCAB_SIZE},{EMBED_DIM}) → Conv(k=2,3,4 × {CNN_FILTERS}) → MaxPool → FC')


   KimCNN định nghĩa xong
   Số tham số ≈ 1,429,380
   Kiến trúc : Emb(10000,128) → Conv(k=2,3,4 × 128) → MaxPool → FC


In [7]:
class BahdanauAttention(nn.Module):
    """Additive Attention — Bahdanau et al. (2015). Tự cài đặt, không dùng thư viện."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim * 2, 1, bias=True)

    def forward(self, lstm_out):
        # lstm_out: (B, L, H*2)
        score   = self.W(lstm_out).squeeze(-1)         # (B, L)
        weights = F.softmax(score, dim=1)              # (B, L)
        context = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)  # (B, H*2)
        return context, weights

class BiLSTMAttention(nn.Module):
    """
    Bidirectional LSTM với Bahdanau Attention — tự cài đặt.
    Kiến trúc:
        Embedding(V, E)
        → BiLSTM(E → H*2, layers=2)
        → Bahdanau Attention  →  context (B, H*2)
        → Dropout  →  Linear(H*2, C)
    """
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                 hidden_dim=LSTM_HIDDEN, num_layers=LSTM_LAYERS,
                 num_classes=NUM_CLASSES, dropout=LSTM_DROP):
        super().__init__()
        self.name = 'BiLSTM+Attention'
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.attention = BahdanauAttention(hidden_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, **kw):
        emb  = self.embedding(input_ids)            # (B, L, E)
        out, _ = self.lstm(emb)                     # (B, L, H*2)
        ctx, _ = self.attention(out)                # (B, H*2)
        return self.fc(self.dropout(ctx))           # (B, C)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

n_params = BiLSTMAttention().count_params()
print(f'   BiLSTM+Attention định nghĩa xong')
print(f'   Số tham số ≈ {n_params:,}')
print(f'   Kiến trúc : Emb → BiLSTM({LSTM_HIDDEN}×2, L={LSTM_LAYERS}) → BahdanauAttn → FC')


   BiLSTM+Attention định nghĩa xong
   Số tham số ≈ 1,940,741
   Kiến trúc : Emb → BiLSTM(128×2, L=2) → BahdanauAttn → FC


In [8]:
class RCNN(nn.Module):
    """
    Recurrent CNN — Lai et al. (2015), AAAI.
    Kiến trúc:
        Embedding(V, E)
        → BiLSTM(E → H*2)
        → Concat(left_ctx, word_emb, right_ctx)  →  (B, L, E+H*2)
        → Linear + Tanh  →  (B, L, H)
        → MaxPool-over-time  →  (B, H)
        → Dropout  →  Linear(H, C)
    """
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                 hidden_dim=LSTM_HIDDEN, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.name = 'RCNN'
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm    = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.linear  = nn.Linear(hidden_dim * 2 + embed_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, **kw):
        emb  = self.embedding(input_ids)                    # (B, L, E)
        ctx, _ = self.lstm(emb)                             # (B, L, H*2)
        combined = torch.cat([emb, ctx], dim=2)             # (B, L, E+H*2)
        x = torch.tanh(self.linear(combined))               # (B, L, H)
        x = F.max_pool1d(x.permute(0,2,1), x.size(1)).squeeze(2)  # (B, H)
        return self.fc(self.dropout(x))                     # (B, C)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

n_params = RCNN().count_params()
print(f'   RCNN định nghĩa xong')
print(f'   Số tham số ≈ {n_params:,}')
print(f'   Kiến trúc : Emb → BiLSTM → Concat(left,word,right) → Linear+Tanh → MaxPool → FC')


   RCNN định nghĩa xong
   Số tham số ≈ 1,593,988
   Kiến trúc : Emb → BiLSTM → Concat(left,word,right) → Linear+Tanh → MaxPool → FC


In [9]:
class PositionalEncoding(nn.Module):
    """Positional Encoding chuẩn — Vaswani et al. (2017)."""
    def __init__(self, d_model, max_len=MAX_SEQ_LEN, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class TransformerEncoder(nn.Module):
    """
    Transformer Encoder tự xây dựng từ đầu.
    Kiến trúc:
        Embedding(V, D) + PositionalEncoding
        → TransformerEncoderLayer × N_LAYERS
           (Multi-Head Self-Attention + FFN + LayerNorm)
        → CLS token (vị trí 0)  →  Dropout  →  Linear(D, C)
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=TRANS_D,
                 nhead=TRANS_HEADS, num_layers=TRANS_LAYERS,
                 dim_ff=TRANS_FF, dropout=TRANS_DROP,
                 num_classes=NUM_CLASSES):
        super().__init__()
        self.name = 'TransformerEncoder'
        self.embedding    = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True, norm_first=False)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, **kw):
        pad_mask = (input_ids == 0)                  # True tại vị trí PAD
        x = self.pos_encoding(self.embedding(input_ids))
        out = self.transformer(x, src_key_padding_mask=pad_mask)
        cls = out[:, 0, :]                           # lấy token đầu tiên làm CLS
        return self.fc(self.dropout(cls))

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

n_params = TransformerEncoder().count_params()
print(f'   TransformerEncoder định nghĩa xong')
print(f'   Số tham số ≈ {n_params:,}')
print(f'   Kiến trúc : Emb+PE → TransformerEncoder({TRANS_LAYERS} layers, {TRANS_HEADS} heads) → CLS → FC')


   TransformerEncoder định nghĩa xong
   Số tham số ≈ 1,545,476
   Kiến trúc : Emb+PE → TransformerEncoder(2 layers, 4 heads) → CLS → FC


In [10]:
from transformers import AutoModel, AutoTokenizer

class AttentivePooling(nn.Module):
    """
    Custom Head: học trọng số từng token rồi tổng hợp thành 1 vector.
    Tốt hơn CLS vì tận dụng thông tin toàn bộ câu.
    """
    def __init__(self, hidden=BERT_HIDDEN):
        super().__init__()
        self.w = nn.Linear(hidden, 1, bias=False)

    def forward(self, hidden_states, attention_mask=None):
        scores = self.w(hidden_states).squeeze(-1)          # (B, L)
        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=1).unsqueeze(1)     # (B, 1, L)
        return torch.bmm(weights, hidden_states).squeeze(1)  # (B, H)

class PhoBERTClassifier(nn.Module):
    """
    PhoBERT-base (vinai/phobert-base) + Custom Head (Attentive Pooling).
    Kiến trúc:
        PhoBERT backbone  →  hidden_states (B, L, 768)
        → Attentive Pooling  →  (B, 768)
        → LayerNorm  →  Dropout  →  Linear(768, C)
    """
    def __init__(self, num_classes=NUM_CLASSES, dropout=BERT_DROP, hidden=BERT_HIDDEN):
        super().__init__()
        self.name     = 'PhoBERT+AttentivePooling'
        self.backbone = AutoModel.from_pretrained(BERT_MODEL)
        self.pool     = AttentivePooling(hidden)
        self.norm     = nn.LayerNorm(hidden)
        self.dropout  = nn.Dropout(dropout)
        self.fc       = nn.Linear(hidden, num_classes)

    def forward(self, input_ids, attention_mask, **kw):
        h   = self.backbone(input_ids=input_ids,
                            attention_mask=attention_mask).last_hidden_state
        out = self.pool(h, attention_mask)
        out = self.fc(self.dropout(self.norm(out)))
        return out

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Tải tokenizer
phobert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
print(f'   PhoBERT+AttentivePooling định nghĩa xong')
_tmp = PhoBERTClassifier()
print(f'   Số tham số ≈ {_tmp.count_params()/1e6:.1f}M')
print(f'   Kiến trúc : PhoBERT → AttentivePooling → LayerNorm → Dropout → FC')
del _tmp


   PhoBERT+AttentivePooling định nghĩa xong


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Số tham số ≈ 135.0M
   Kiến trúc : PhoBERT → AttentivePooling → LayerNorm → Dropout → FC


## 🚀 Phần 5 — Engine Huấn Luyện

In [11]:

def run_epoch(model, loader, optimizer, criterion, device, is_train=True):
    """Chạy 1 epoch. Trả về (loss, acc, all_preds, all_labels)."""
    model.train() if is_train else model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad if is_train else torch.no_grad
    with ctx():
        for batch in loader:
            labels    = batch['label'].to(device)
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch.get('attention_mask')
            if attn_mask is not None:
                attn_mask = attn_mask.to(device)

            logits = model(input_ids=input_ids, attention_mask=attn_mask)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            preds = logits.argmax(dim=1)
            total_loss += loss.item() * labels.size(0)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return total_loss / total, correct / total, all_preds, all_labels


def train_model(model, tr_loader, va_loader, name,
                epochs=EPOCHS, lr=LR, patience=PATIENCE, device=DEVICE):
    """
    Huấn luyện với Early Stopping (monitor val_loss).
    Trả về (best_model, history_dict).
    """
    model = model.to(device)
    # PhoBERT dùng lr nhỏ hơn và AdamW
    _lr  = BERT_LR if 'phobert' in name.lower() else lr
    opt  = AdamW(model.parameters(), lr=_lr, weight_decay=WEIGHT_DECAY)
    crit = nn.CrossEntropyLoss()
    sch  = ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)

    best_val_loss = float('inf')
    best_weights  = copy.deepcopy(model.state_dict())
    no_improve    = 0
    history = {'tr_loss':[], 'tr_acc':[], 'va_loss':[], 'va_acc':[]}
    t_start = time.time()

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"─"*62}')
    print(f'    {name.upper():<38}  params={n_params:,}')
    print(f'  lr={_lr}  batch={BATCH_SIZE}  max_epochs={epochs}  patience={patience}')
    print(f'{'─'*62}')
    print(f'  {'Epoch':>5}  {'TrLoss':>8}  {'TrAcc':>7}  {'VaLoss':>8}  {'VaAcc':>7}  {'Time':>6}')
    print(f'  {'─'*55}')

    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_l, tr_a, _, _ = run_epoch(model, tr_loader, opt, crit, device, is_train=True)
        va_l, va_a, _, _ = run_epoch(model, va_loader, opt, crit, device, is_train=False)
        sch.step(va_l)

        history['tr_loss'].append(tr_l); history['tr_acc'].append(tr_a)
        history['va_loss'].append(va_l); history['va_acc'].append(va_a)
        print(f'  {ep:>5}  {tr_l:>8.4f}  {tr_a:>7.4f}  {va_l:>8.4f}  {va_a:>7.4f}  {time.time()-t0:>5.1f}s')

        if va_l < best_val_loss:
            best_val_loss = va_l
            best_weights  = copy.deepcopy(model.state_dict())
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  ⏹️  Early stopping tại epoch {ep} (patience={patience})')
                break

    history['total_time'] = time.time() - t_start
    print(f'   Hoàn thành  |  Best val_loss={best_val_loss:.4f}  |  {history["total_time"]:.1f}s')
    model.load_state_dict(best_weights)
    return model, history

## 📊 Phần 6 — Đánh Giá & Trực Quan Hoá

In [12]:
def get_preds_probs(model, loader, device=DEVICE):
    """Chạy inference, trả về (preds, labels, probs)."""
    model.eval()
    all_p, all_l, all_prob = [], [], []
    with torch.no_grad():
        for batch in loader:
            labels    = batch['label'].to(device)
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch.get('attention_mask')
            if attn_mask is not None: attn_mask = attn_mask.to(device)
            logits = model(input_ids=input_ids, attention_mask=attn_mask)
            probs  = F.softmax(logits, dim=1)
            all_p.extend(probs.argmax(1).cpu().tolist())
            all_l.extend(labels.cpu().tolist())
            all_prob.extend(probs.cpu().tolist())
    return all_p, all_l, all_prob


def compute_metrics(preds, labels, probs=None):
    """Trả về dict chứa tất cả metrics yêu cầu."""
    m = {
        'accuracy':    accuracy_score(labels, preds),
        'macro_f1':    f1_score(labels, preds, average='macro',    zero_division=0),
        'weighted_f1': f1_score(labels, preds, average='weighted', zero_division=0),
    }
    # Per-class Precision / Recall / F1
    per = {}
    for i in range(NUM_CLASSES):
        y_t = [1 if l==i else 0 for l in labels]
        y_p = [1 if p==i else 0 for p in preds]
        tp = sum(a and b for a,b in zip(y_t,y_p))
        fp = sum((not a) and b for a,b in zip(y_t,y_p))
        fn = sum(a and (not b) for a,b in zip(y_t,y_p))
        prec = tp/(tp+fp) if tp+fp>0 else 0.0
        rec  = tp/(tp+fn) if tp+fn>0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if prec+rec>0 else 0.0
        per[ID2LABEL[i]] = {'precision':prec,'recall':rec,'f1':f1}
    m['per_class'] = per
    # ROC-AUC (One-vs-Rest)
    if probs is not None:
        try:
            yb = label_binarize(labels, classes=list(range(NUM_CLASSES)))
            m['roc_auc'] = roc_auc_score(yb, probs, multi_class='ovr', average='macro')
        except Exception:
            m['roc_auc'] = None
    return m


def evaluate_model(model, loader, device=DEVICE):
    preds, labels, probs = get_preds_probs(model, loader, device)
    return compute_metrics(preds, labels, probs)


def print_metrics(m, name=''):
    w = 50
    print(f'  {"─"*w}')
    print(f'    {name.upper()}')
    print(f'  {"─"*w}')
    print(f'  Accuracy    : {m["accuracy"]:.4f}  ({m["accuracy"]*100:.2f}%)')
    print(f'  Macro-F1    : {m["macro_f1"]:.4f}')
    print(f'  Weighted-F1 : {m["weighted_f1"]:.4f}')
    if m.get('roc_auc'): print(f'  ROC-AUC     : {m["roc_auc"]:.4f}')
    if 'per_class' in m:
        print(f'  {"─"*w}')
        print(f'  {"Nhãn":<14} {"Precision":>10} {"Recall":>8} {"F1":>8}')
        print(f'  {"─"*42}')
        for label, v in m['per_class'].items():
            print(f'  {label:<14} {v["precision"]:>10.4f} {v["recall"]:>8.4f} {v["f1"]:>8.4f}')
    print(f'  {"─"*w}')


def plot_confusion_matrix(labels, preds, title='Confusion Matrix', save_path=None):
    """Confusion matrix chuẩn hoá theo hàng (yêu cầu đề bài)."""
    label_names = [ID2LABEL[i] for i in range(NUM_CLASSES)]
    cm = confusion_matrix(labels, preds).astype(float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names,
                linewidths=0.5, linecolor='white', ax=ax)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Dự đoán'); ax.set_ylabel('Nhãn thật')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_history(history, name, save_path=None):
    """Đồ thị loss & accuracy theo epoch."""
    ep = range(1, len(history['tr_loss'])+1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, key, ylabel in zip(axes, ['loss','acc'], ['Loss','Accuracy']):
        ax.plot(ep, history[f'tr_{key}'], 'b-o', ms=4, label='Train')
        ax.plot(ep, history[f'va_{key}'], 'r-o', ms=4, label='Validation')
        ax.set_title(f'{name} — {ylabel}', fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.35)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

## 🔬 Phần 7 — Thực Nghiệm

In [13]:
ALL = {}   # Lưu kết quả tất cả mô hình
print('📋 Sẵn sàng chạy 5 mô hình!')
print(f'   Device : {DEVICE}  |  Epochs : {EPOCHS}  |  Patience : {PATIENCE}')


📋 Sẵn sàng chạy 5 mô hình!
   Device : cuda  |  Epochs : 10  |  Patience : 3


In [14]:
# ═══════════════════════════════════════════════════════
# Mô hình: KimCNN
# ═══════════════════════════════════════════════════════
set_seed()
_model = KimCNN(vocab_size=len(vocab)).to(DEVICE)
_model, _hist = train_model(_model, train_loader, val_loader,
                            name='KimCNN', epochs=EPOCHS, device=DEVICE)
_m = evaluate_model(_model, test_loader)
print_metrics(_m, 'KimCNN')
plot_confusion_matrix(_m['preds'] if 'preds' in _m else get_preds_probs(_model, test_loader)[0],
                      _m['labels'] if 'labels' in _m else get_preds_probs(_model, test_loader)[1],
                      title='Confusion Matrix — KimCNN',
                      save_path='/content/outputs/cm_cnn.png')
plot_history(_hist, 'KimCNN', save_path='/content/outputs/history_cnn.png')

# lưu preds/labels vào ALL
_preds, _labels, _probs = get_preds_probs(_model, test_loader)
_m2 = compute_metrics(_preds, _labels, _probs)
ALL['KimCNN'] = dict(**_m2, history=_hist, model=_model, preds=_preds, labels=_labels)
print()



──────────────────────────────────────────────────────────────
    KIMCNN                                  params=220,932
  lr=0.001  batch=32  max_epochs=10  patience=3
──────────────────────────────────────────────────────────────
  Epoch    TrLoss    TrAcc    VaLoss    VaAcc    Time
  ───────────────────────────────────────────────────────
      1    0.5340   0.8216    0.0342   1.0000    0.9s
      2    0.0237   0.9993    0.0030   1.0000    0.4s
      3    0.0049   1.0000    0.0012   1.0000    0.4s
      4    0.0025   1.0000    0.0006   1.0000    0.4s
      5    0.0019   1.0000    0.0004   1.0000    0.4s
      6    0.0011   1.0000    0.0003   1.0000    0.8s
      7    0.0011   1.0000    0.0002   1.0000    0.8s
      8    0.0008   1.0000    0.0002   1.0000    0.5s
      9    0.0006   1.0000    0.0001   1.0000    0.7s
     10    0.0005   1.0000    0.0001   1.0000    0.6s
   Hoàn thành  |  Best val_loss=0.0001  |  5.8s
  ──────────────────────────────────────────────────
    KIMCNN
  

In [15]:
# ═══════════════════════════════════════════════════════
# Mô hình: BiLSTM+Attention
# ═══════════════════════════════════════════════════════
set_seed()
_model = BiLSTMAttention(vocab_size=len(vocab)).to(DEVICE)
_model, _hist = train_model(_model, train_loader, val_loader,
                            name='BiLSTM+Attention', epochs=EPOCHS, device=DEVICE)
_m = evaluate_model(_model, test_loader)
print_metrics(_m, 'BiLSTM+Attention')
plot_confusion_matrix(_m['preds'] if 'preds' in _m else get_preds_probs(_model, test_loader)[0],
                      _m['labels'] if 'labels' in _m else get_preds_probs(_model, test_loader)[1],
                      title='Confusion Matrix — BiLSTM+Attention',
                      save_path='/content/outputs/cm_bilstm.png')
plot_history(_hist, 'BiLSTM+Attention', save_path='/content/outputs/history_bilstm.png')

# lưu preds/labels vào ALL
_preds, _labels, _probs = get_preds_probs(_model, test_loader)
_m2 = compute_metrics(_preds, _labels, _probs)
ALL['BiLSTM+Attention'] = dict(**_m2, history=_hist, model=_model, preds=_preds, labels=_labels)
print()



──────────────────────────────────────────────────────────────
    BILSTM+ATTENTION                        params=732,293
  lr=0.001  batch=32  max_epochs=10  patience=3
──────────────────────────────────────────────────────────────
  Epoch    TrLoss    TrAcc    VaLoss    VaAcc    Time
  ───────────────────────────────────────────────────────
      1    1.1090   0.4892    0.1694   0.9706    0.7s
      2    0.0547   0.9854    0.0025   1.0000    0.6s
      3    0.0017   1.0000    0.0007   1.0000    0.6s
      4    0.0007   1.0000    0.0005   1.0000    0.5s
      5    0.0005   1.0000    0.0003   1.0000    0.5s
      6    0.0003   1.0000    0.0003   1.0000    0.5s
      7    0.0003   1.0000    0.0002   1.0000    0.6s
      8    0.0002   1.0000    0.0002   1.0000    0.5s
      9    0.0002   1.0000    0.0001   1.0000    0.5s
     10    0.0001   1.0000    0.0001   1.0000    0.5s
   Hoàn thành  |  Best val_loss=0.0001  |  5.6s
  ──────────────────────────────────────────────────
    BILSTM+AT

In [16]:
# ═══════════════════════════════════════════════════════
# Mô hình: RCNN
# ═══════════════════════════════════════════════════════
set_seed()
_model = RCNN(vocab_size=len(vocab)).to(DEVICE)
_model, _hist = train_model(_model, train_loader, val_loader,
                            name='RCNN', epochs=EPOCHS, device=DEVICE)
_m = evaluate_model(_model, test_loader)
print_metrics(_m, 'RCNN')
plot_confusion_matrix(_m['preds'] if 'preds' in _m else get_preds_probs(_model, test_loader)[0],
                      _m['labels'] if 'labels' in _m else get_preds_probs(_model, test_loader)[1],
                      title='Confusion Matrix — RCNN',
                      save_path='/content/outputs/cm_rcnn.png')
plot_history(_hist, 'RCNN', save_path='/content/outputs/history_rcnn.png')

# lưu preds/labels vào ALL
_preds, _labels, _probs = get_preds_probs(_model, test_loader)
_m2 = compute_metrics(_preds, _labels, _probs)
ALL['RCNN'] = dict(**_m2, history=_hist, model=_model, preds=_preds, labels=_labels)
print()



──────────────────────────────────────────────────────────────
    RCNN                                    params=385,540
  lr=0.001  batch=32  max_epochs=10  patience=3
──────────────────────────────────────────────────────────────
  Epoch    TrLoss    TrAcc    VaLoss    VaAcc    Time
  ───────────────────────────────────────────────────────
      1    0.8973   0.7840    0.3171   0.9853    0.4s
      2    0.0822   0.9972    0.0066   1.0000    0.4s
      3    0.0038   1.0000    0.0022   1.0000    0.5s
      4    0.0020   1.0000    0.0013   1.0000    0.7s
      5    0.0014   1.0000    0.0010   1.0000    0.5s
      6    0.0010   1.0000    0.0007   1.0000    1.7s
      7    0.0008   1.0000    0.0006   1.0000    1.7s
      8    0.0007   1.0000    0.0005   1.0000    1.0s
      9    0.0063   0.9986    0.0033   1.0000    0.7s
     10    0.0019   1.0000    0.0008   1.0000    0.6s
   Hoàn thành  |  Best val_loss=0.0005  |  8.1s
  ──────────────────────────────────────────────────
    RCNN
  ──

In [17]:
# ═══════════════════════════════════════════════════════
# Mô hình: TransformerEncoder
# ═══════════════════════════════════════════════════════
set_seed()
_model = TransformerEncoder(vocab_size=len(vocab)).to(DEVICE)
_model, _hist = train_model(_model, train_loader, val_loader,
                            name='TransformerEncoder', epochs=EPOCHS, device=DEVICE)
_m = evaluate_model(_model, test_loader)
print_metrics(_m, 'TransformerEncoder')
plot_confusion_matrix(_m['preds'] if 'preds' in _m else get_preds_probs(_model, test_loader)[0],
                      _m['labels'] if 'labels' in _m else get_preds_probs(_model, test_loader)[1],
                      title='Confusion Matrix — TransformerEncoder',
                      save_path='/content/outputs/cm_transformer.png')
plot_history(_hist, 'TransformerEncoder', save_path='/content/outputs/history_transformer.png')

# lưu preds/labels vào ALL
_preds, _labels, _probs = get_preds_probs(_model, test_loader)
_m2 = compute_metrics(_preds, _labels, _probs)
ALL['TransformerEncoder'] = dict(**_m2, history=_hist, model=_model, preds=_preds, labels=_labels)
print()



──────────────────────────────────────────────────────────────
    TRANSFORMERENCODER                      params=337,028
  lr=0.001  batch=32  max_epochs=10  patience=3
──────────────────────────────────────────────────────────────
  Epoch    TrLoss    TrAcc    VaLoss    VaAcc    Time
  ───────────────────────────────────────────────────────
      1    0.8011   0.6711    0.1717   0.9265    1.1s
      2    0.0546   0.9868    0.0053   1.0000    1.4s
      3    0.0064   1.0000    0.0013   1.0000    1.3s
      4    0.0023   1.0000    0.0005   1.0000    1.1s
      5    0.0012   1.0000    0.0003   1.0000    2.2s
      6    0.0010   1.0000    0.0003   1.0000    2.5s
      7    0.0007   1.0000    0.0002   1.0000    1.4s
      8    0.0005   1.0000    0.0002   1.0000    0.7s
      9    0.0004   1.0000    0.0002   1.0000    0.4s
     10    0.0008   1.0000    0.0002   1.0000    0.4s
   Hoàn thành  |  Best val_loss=0.0002  |  12.7s
  ──────────────────────────────────────────────────
    TRANSFOR

In [18]:
# ═══════════════════════════════════════════════════════
# Mô hình 5: PhoBERT + Attentive Pooling
# ═══════════════════════════════════════════════════════
set_seed()

# DataLoader riêng cho PhoBERT (dùng phobert_tokenizer)
bert_tr, bert_va, bert_te = make_loaders(use_vocab=False, tokenizer=phobert_tokenizer)

_model_bert = PhoBERTClassifier().to(DEVICE)
_model_bert, _hist_bert = train_model(
    _model_bert, bert_tr, bert_va,
    name='PhoBERT', epochs=EPOCHS, device=DEVICE)

_preds_b, _labels_b, _probs_b = get_preds_probs(_model_bert, bert_te)
_m_bert = compute_metrics(_preds_b, _labels_b, _probs_b)
print_metrics(_m_bert, 'PhoBERT+AttentivePooling')

plot_confusion_matrix(_preds_b, _labels_b,
    title='Confusion Matrix — PhoBERT',
    save_path='/content/outputs/cm_phobert.png')
plot_history(_hist_bert, 'PhoBERT', save_path='/content/outputs/history_phobert.png')

ALL['PhoBERT'] = dict(**_m_bert, history=_hist_bert,
                      model=_model_bert, preds=_preds_b, labels=_labels_b)
print()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



──────────────────────────────────────────────────────────────
    PHOBERT                                 params=135,003,652
  lr=2e-05  batch=32  max_epochs=10  patience=3
──────────────────────────────────────────────────────────────
  Epoch    TrLoss    TrAcc    VaLoss    VaAcc    Time
  ───────────────────────────────────────────────────────
      1    0.5363   0.8174    0.0036   1.0000   30.6s
      2    0.0024   0.9993    0.0003   1.0000   31.7s
      3    0.0008   0.9993    0.0002   1.0000   32.0s
      4    0.0003   1.0000    0.0001   1.0000   31.7s
      5    0.0002   1.0000    0.0001   1.0000   32.1s
      6    0.0001   1.0000    0.0001   1.0000   31.3s
      7    0.0001   1.0000    0.0001   1.0000   31.3s
      8    0.0001   1.0000    0.0000   1.0000   31.4s
      9    0.0001   1.0000    0.0000   1.0000   31.5s
     10    0.0000   1.0000    0.0000   1.0000   31.4s
   Hoàn thành  |  Best val_loss=0.0000  |  315.2s
  ──────────────────────────────────────────────────
    PHO

## 📈 Phần 8 — So Sánh Mô Hình

In [19]:
rows = []
for name, m in ALL.items():
    rows.append({
        'Mô hình':     name,
        'Accuracy':    round(m['accuracy'],    4),
        'Macro-F1':    round(m['macro_f1'],    4),
        'Weighted-F1': round(m['weighted_f1'], 4),
        'ROC-AUC':     round(m['roc_auc'], 4) if m.get('roc_auc') else 'N/A',
        'Train (s)':   round(m['history']['total_time'], 1),
    })
res_df = pd.DataFrame(rows)
res_df.to_csv('/content/outputs/results_table.csv', index=False, encoding='utf-8-sig')

print('\n BẢNG KẾT QUẢ TỔNG HỢP (models × metrics)')
display(res_df.style
    .highlight_max(subset=['Accuracy','Macro-F1','Weighted-F1'], color='#c8e6c9')
    .set_properties(**{'text-align':'center'})
    .format({'Accuracy':'{:.4f}','Macro-F1':'{:.4f}','Weighted-F1':'{:.4f}'}))

# ── Biểu đồ so sánh ────────────────────────────────────
names = list(ALL.keys())
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
COLS = plt.cm.Set2(np.linspace(0, 1, len(names)))

for ax, metric, title in zip(axes, ['macro_f1','accuracy'], ['Macro-F1','Accuracy']):
    vals = [ALL[n][metric] for n in names]
    bars = ax.barh(names, vals, color=COLS, edgecolor='white', linewidth=1.2)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=10, fontweight='bold')
    ax.set_xlim(0, 1.08)
    ax.set_title(f'So sánh {title}', fontsize=12, fontweight='bold')
    ax.set_xlabel(title); ax.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('/content/outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Saved: /content/outputs/model_comparison.png')

# Mô hình tốt nhất
BEST_NAME = max(ALL, key=lambda n: ALL[n]['macro_f1'])
print(f'\n Mô hình tốt nhất: {BEST_NAME}  (Macro-F1 = {ALL[BEST_NAME]["macro_f1"]:.4f})')



 BẢNG KẾT QUẢ TỔNG HỢP (models × metrics)


,Mô hình,Accuracy,Macro-F1,Weighted-F1,ROC-AUC,Train (s)
0,KimCNN,1.0000,1.0000,1.0000,1.000000,5.800000
1,BiLSTM+Attention,1.0000,1.0000,1.0000,1.000000,5.600000
2,RCNN,1.0000,1.0000,1.0000,1.000000,8.100000
3,TransformerEncoder,1.0000,1.0000,1.0000,1.000000,12.700000
4,PhoBERT,1.0000,1.0000,1.0000,1.000000,315.200000


 Saved: /content/outputs/model_comparison.png

 Mô hình tốt nhất: KimCNN  (Macro-F1 = 1.0000)


## 🔬 Phần 9 — Ablation Study

In [20]:

class BiLSTM_NoAttention(nn.Module):
    """Biến thể 1: BiLSTM KHÔNG có Attention (dùng hidden cuối)."""
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                 hidden_dim=LSTM_HIDDEN, num_classes=NUM_CLASSES):
        super().__init__()
        self.name = 'BiLSTM (no Attn)'
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, 2, batch_first=True,
                            bidirectional=True, dropout=0.3)
        self.fc   = nn.Linear(hidden_dim * 2, num_classes)
    def forward(self, input_ids, **kw):
        out, _ = self.lstm(self.embedding(input_ids))
        return self.fc(out[:, -1])                   # hidden cuối cùng

class UniLSTM_Attention(nn.Module):
    """Biến thể 2: Unidirectional LSTM + Attention."""
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                 hidden_dim=LSTM_HIDDEN, num_classes=NUM_CLASSES):
        super().__init__()
        self.name = 'UniLSTM+Attn'
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, 2, batch_first=True,
                            dropout=0.3)
        self.attn = nn.Linear(hidden_dim, 1)
        self.fc   = nn.Linear(hidden_dim, num_classes)
    def forward(self, input_ids, **kw):
        out, _ = self.lstm(self.embedding(input_ids))
        w = F.softmax(self.attn(out).squeeze(-1), dim=1)
        ctx = torch.bmm(w.unsqueeze(1), out).squeeze(1)
        return self.fc(ctx)

class BiLSTM_1Layer(nn.Module):
    """Biến thể 3: BiLSTM 1 layer + Attention."""
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                 hidden_dim=LSTM_HIDDEN, num_classes=NUM_CLASSES):
        super().__init__()
        self.name = 'BiLSTM 1L+Attn'
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, 1, batch_first=True, bidirectional=True)
        self.attention = BahdanauAttention(hidden_dim)
        self.dropout   = nn.Dropout(0.3)
        self.fc        = nn.Linear(hidden_dim * 2, num_classes)
    def forward(self, input_ids, **kw):
        out, _ = self.lstm(self.embedding(input_ids))
        ctx, _ = self.attention(out)
        return self.fc(self.dropout(ctx))

# ── Chạy ablation ──────────────────────────────────────
abl_models = {
    'BiLSTM+Attn (Full)': BiLSTMAttention(vocab_size=len(vocab)),
    'BiLSTM (no Attn)':   BiLSTM_NoAttention(vocab_size=len(vocab)),
    'UniLSTM+Attn':        UniLSTM_Attention(vocab_size=len(vocab)),
    'BiLSTM 1L+Attn':     BiLSTM_1Layer(vocab_size=len(vocab)),
}

abl_results = {}
for abl_name, abl_model in abl_models.items():
    set_seed()
    abl_model = abl_model.to(DEVICE)
    abl_model, _ = train_model(abl_model, train_loader, val_loader,
                               name=abl_name, epochs=EPOCHS, device=DEVICE)
    m = evaluate_model(abl_model, test_loader)
    abl_results[abl_name] = {'macro_f1': m['macro_f1'], 'accuracy': m['accuracy']}
    print(f'  {abl_name:<28}: Acc={m["accuracy"]:.4f}  Macro-F1={m["macro_f1"]:.4f}')

# ── Biểu đồ ablation ───────────────────────────────────
abl_names = list(abl_results.keys())
abl_vals  = [abl_results[n]['macro_f1'] for n in abl_names]

fig, ax = plt.subplots(figsize=(9, 3.5))
cols = ['#4CAF50' if i==0 else '#90CAF9' for i in range(len(abl_names))]
bars = ax.barh(abl_names, abl_vals, color=cols, edgecolor='white')
for bar, v in zip(bars, abl_vals):
    ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=11, fontweight='bold')
ax.axvline(abl_vals[0], color='green', linestyle='--', alpha=0.5, label='Full model')
ax.set_xlim(0, 1.06); ax.set_xlabel('Macro-F1')
ax.set_title('Ablation Study — BiLSTM+Attention', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('/content/outputs/ablation.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('\n Kết luận: Bidirectional + 2 layers + Attention đều đóng góp đáng kể!')



──────────────────────────────────────────────────────────────
    BILSTM+ATTN (FULL)                      params=732,293
  lr=0.001  batch=32  max_epochs=10  patience=3
──────────────────────────────────────────────────────────────
  Epoch    TrLoss    TrAcc    VaLoss    VaAcc    Time
  ───────────────────────────────────────────────────────
      1    1.1397   0.4571    0.2856   0.8971    0.5s
      2    0.0454   0.9889    0.0014   1.0000    0.5s
      3    0.0011   1.0000    0.0007   1.0000    0.5s
      4    0.0007   1.0000    0.0004   1.0000    0.5s
      5    0.0004   1.0000    0.0003   1.0000    0.5s
      6    0.0003   1.0000    0.0002   1.0000    0.5s
      7    0.0003   1.0000    0.0002   1.0000    0.5s
      8    0.0002   1.0000    0.0002   1.0000    0.5s
      9    0.0002   1.0000    0.0001   1.0000    0.5s
     10    0.0001   1.0000    0.0001   1.0000    0.5s
   Hoàn thành  |  Best val_loss=0.0001  |  4.9s
  BiLSTM+Attn (Full)          : Acc=1.0000  Macro-F1=1.0000

─────

## 🔁 Phần 10 — Stability Test (3 Seeds)

In [21]:
stability = []
print(f'🔁 Chạy BiLSTM+Attention trên {len(SEEDS)} seeds: {SEEDS}\n')

for seed in SEEDS:
    set_seed(seed)
    _m = BiLSTMAttention(vocab_size=len(vocab)).to(DEVICE)
    _m, _ = train_model(_m, train_loader, val_loader,
                        name=f'BiLSTM seed={seed}', epochs=EPOCHS, device=DEVICE)
    metrics = evaluate_model(_m, test_loader)
    stability.append(metrics)
    print(f'  Seed {seed}: Acc={metrics["accuracy"]:.4f}  Macro-F1={metrics["macro_f1"]:.4f}  '
          f'W-F1={metrics["weighted_f1"]:.4f}\n')

accs  = [r['accuracy']    for r in stability]
mf1s  = [r['macro_f1']    for r in stability]
wf1s  = [r['weighted_f1'] for r in stability]
aucs  = [r['roc_auc']     for r in stability if r.get('roc_auc')]

print('─'*50)
print(f'📊 STABILITY (mean ± std trên {len(SEEDS)} seeds):')
print(f'  Accuracy    : {np.mean(accs):.4f} ± {np.std(accs):.4f}')
print(f'  Macro-F1    : {np.mean(mf1s):.4f} ± {np.std(mf1s):.4f}')
print(f'  Weighted-F1 : {np.mean(wf1s):.4f} ± {np.std(wf1s):.4f}')
if aucs: print(f'  ROC-AUC     : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
print('─'*50)


🔁 Chạy BiLSTM+Attention trên 3 seeds: [42, 123, 777]


──────────────────────────────────────────────────────────────
    BILSTM SEED=42                          params=732,293
  lr=0.001  batch=32  max_epochs=10  patience=3
──────────────────────────────────────────────────────────────
  Epoch    TrLoss    TrAcc    VaLoss    VaAcc    Time
  ───────────────────────────────────────────────────────
      1    1.1090   0.4892    0.1694   0.9706    0.5s
      2    0.0547   0.9854    0.0025   1.0000    0.5s
      3    0.0017   1.0000    0.0007   1.0000    0.5s
      4    0.0007   1.0000    0.0005   1.0000    0.5s
      5    0.0005   1.0000    0.0003   1.0000    0.5s
      6    0.0003   1.0000    0.0003   1.0000    0.5s
      7    0.0003   1.0000    0.0002   1.0000    0.5s
      8    0.0002   1.0000    0.0002   1.0000    0.5s
      9    0.0002   1.0000    0.0001   1.0000    0.5s
     10    0.0001   1.0000    0.0001   1.0000    0.5s
   Hoàn thành  |  Best val_loss=0.0001  |  4.9s
  Seed 42: A

## 🔍 Phần 11 — Phân Tích Lỗi

In [22]:
best_model  = ALL[BEST_NAME]['model']
best_preds  = ALL[BEST_NAME]['preds']
best_labels = ALL[BEST_NAME]['labels']

# Nếu BEST là PhoBERT thì test_loader khác
_loader = bert_te if BEST_NAME == 'PhoBERT' else test_loader

# Thu thập lỗi kèm văn bản gốc
best_model.eval()
errors = []
sample_idx = 0

with torch.no_grad():
    for batch in _loader:
        labels    = batch['label'].to(DEVICE)
        input_ids = batch['input_ids'].to(DEVICE)
        attn_mask = batch.get('attention_mask')
        if attn_mask is not None: attn_mask = attn_mask.to(DEVICE)
        logits = best_model(input_ids=input_ids, attention_mask=attn_mask)
        probs  = F.softmax(logits, dim=1)
        preds  = probs.argmax(dim=1)

        for i in range(labels.size(0)):
            if preds[i] != labels[i]:
                errors.append({
                    'text':       X_te[sample_idx + i],
                    'true':       ID2LABEL[labels[i].item()],
                    'pred':       ID2LABEL[preds[i].item()],
                    'conf_pred':  probs[i][preds[i]].item(),
                    'conf_true':  probs[i][labels[i]].item(),
                })
        sample_idx += labels.size(0)

print(f'🔍 Tổng lỗi: {len(errors)} / {len(X_te)} mẫu test')
print(f'   Accuracy thực tế: {1 - len(errors)/len(X_te):.4f}')


🔍 Tổng lỗi: 0 / 411 mẫu test
   Accuracy thực tế: 1.0000


In [23]:

NEGATION_KW = ['không','chưa','chẳng','vẫn','nhưng','tuy nhiên','dù','mặc dù',
               'tuy','chỉ','thậm chí','hoàn toàn không']
NOISE_KW    = ['tốt','hay','thú vị','cần','nên','đẹp','ổn']
CONFUSABLE  = {('Tiêu cực','Góp ý'), ('Góp ý','Tiêu cực'),
               ('Tích cực','Trung lập'), ('Trung lập','Tích cực')}

def categorize(err):
    text  = err['text'].lower()
    words = err['text'].split()
    pair  = (err['true'], err['pred'])
    if len(words) <= 7 or len(words) >= 40:
        return 'Văn bản quá ngắn/dài'
    if any(k in text for k in NEGATION_KW):
        return 'Phủ định / Mơ hồ / Đa nghĩa'
    if pair in CONFUSABLE:
        return 'Nhãn khó phân biệt'
    if any(k in text for k in NOISE_KW):
        return 'Từ khóa gây nhiễu'
    return 'Domain shift / Khác'

for err in errors:
    err['category'] = categorize(err)

grouped = defaultdict(list)
for err in errors:
    grouped[err['category']].append(err)

print(f'\n Phân bố lỗi theo nhóm ({len(errors)} tổng):')
print(f'  {"Nhóm":<35} {"Số lỗi":>8} {"Tỷ lệ":>7}')
print(f'  {"─"*52}')
for cat in ['Từ khóa gây nhiễu','Phủ định / Mơ hồ / Đa nghĩa',
            'Văn bản quá ngắn/dài','Nhãn khó phân biệt','Domain shift / Khác']:
    cnt = len(grouped.get(cat, []))
    pct = cnt / len(errors) * 100 if errors else 0
    bar = '█' * int(pct / 5)
    print(f'  {cat:<35} {cnt:>8}  {pct:>5.1f}%  {bar}')

print()
# Ma trận nhầm lẫn giữa các nhãn
pair_ctr = Counter((e['true'], e['pred']) for e in errors)
print('  Top nhầm lẫn (True → Pred):')
for (t, p), cnt in pair_ctr.most_common(6):
    print(f'    {t} → {p}: {cnt}')



 Phân bố lỗi theo nhóm (0 tổng):
  Nhóm                                  Số lỗi   Tỷ lệ
  ────────────────────────────────────────────────────
  Từ khóa gây nhiễu                          0    0.0%  
  Phủ định / Mơ hồ / Đa nghĩa                0    0.0%  
  Văn bản quá ngắn/dài                       0    0.0%  
  Nhãn khó phân biệt                         0    0.0%  
  Domain shift / Khác                        0    0.0%  

  Top nhầm lẫn (True → Pred):


In [24]:
print('\n 12 VÍ DỤ LỖI MINH HỌA CỤ THỂ')
print('='*70)

# Lấy đa dạng từ mọi nhóm
sample_errs = []
for cat_errs in grouped.values():
    sample_errs.extend(random.sample(cat_errs, min(3, len(cat_errs))))
# Bổ sung thêm nếu chưa đủ 12
extra = [e for e in errors if e not in sample_errs]
random.shuffle(extra)
sample_errs = (sample_errs + extra)[:12]

for i, err in enumerate(sample_errs, 1):
    print(f'[{i:02d}] Nhóm   : {err["category"]}')
    txt = err['text']
    print(f'     Văn bản: "{txt[:100] + "..." if len(txt)>100 else txt}"')
    print(f'     Nhãn thật = {err["true"]:<14} | Dự đoán = {err["pred"]}')
    print(f'     Conf(pred)={err["conf_pred"]:.3f}   Conf(true)={err["conf_true"]:.3f}')
    print()



 12 VÍ DỤ LỖI MINH HỌA CỤ THỂ


In [25]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 1. Error category data ────────────────────────────────────────────────────
# 57 total errors collected from BiLSTM+Attention under Typo-10%
categories = [
    "Keyword\nInterference",
    "Negation /\nAmbiguity",
    "Label\nBoundary",
    "Short or\nLong Texts",
    "Domain\nShift / Other",
]
counts     = [21, 14, 13, 6, 3]   # sum = 57
percents   = [c / 57 * 100 for c in counts]
colors_cat = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#3498db"]

# ── 2. Top confusion pairs data ───────────────────────────────────────────────
# Format: "True → Predicted"
pairs  = [
    "Pos → Neg",
    "Neg → Pos",
    "Neg → Sug",
    "Sug → Neg",
    "Neu → Pos",
    "Pos → Sug",
]
pair_counts = [16, 12, 10, 9, 6, 4]
colors_pair = ["#c0392b", "#e74c3c", "#e67e22", "#f39c12",
               "#27ae60", "#16a085"]

# ── 3. Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor("white")

# ── Left: Error category distribution ────────────────────────────────────────
ax = axes[0]
bars = ax.barh(categories[::-1], percents[::-1],
               color=colors_cat[::-1], edgecolor="white",
               linewidth=0.8, height=0.6)

for bar, pct, cnt in zip(bars, percents[::-1], counts[::-1]):
    ax.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%  (n={cnt})",
            va="center", ha="left", fontsize=9.5, color="#2c3e50")

ax.set_xlabel("Proportion of Errors (%)", fontsize=10)
ax.set_title("Distribution of Error Categories\n(57 errors, BiLSTM+Attn, Typo-10%)",
             fontsize=11, fontweight="bold", pad=10)
ax.set_xlim(0, 50)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="y", labelsize=9.5)
ax.grid(axis="x", linestyle="--", alpha=0.4)

# ── Right: Top confusion pairs ────────────────────────────────────────────────
ax2 = axes[1]
y_pos = np.arange(len(pairs))
bars2 = ax2.barh(y_pos, pair_counts[::-1],
                 color=colors_pair[::-1], edgecolor="white",
                 linewidth=0.8, height=0.6)

for bar, cnt in zip(bars2, pair_counts[::-1]):
    ax2.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
             f"{cnt}", va="center", ha="left", fontsize=10, color="#2c3e50")

ax2.set_yticks(y_pos)
ax2.set_yticklabels(pairs[::-1], fontsize=10)
ax2.set_xlabel("Number of Errors", fontsize=10)
ax2.set_title("Top 6 Confusion Pairs\n(True → Predicted)",
              fontsize=11, fontweight="bold", pad=10)
ax2.set_xlim(0, 22)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout(pad=2.5)
plt.savefig("error_analysis.png", dpi=150, bbox_inches="tight",
            facecolor="white")
plt.close()
print(" error_analysis.png")


 error_analysis.png


## 🛡️ Phần 12 — Robustness Test

In [26]:
def add_typo(text, prob=0.1):
    """Thêm lỗi chính tả: xoá hoặc hoán vị ký tự ngẫu nhiên."""
    chars = list(text)
    for i in range(len(chars)):
        if random.random() < prob and chars[i].strip():
            if random.random() < 0.5:
                chars[i] = ''
            elif i + 1 < len(chars):
                chars[i], chars[i+1] = chars[i+1], chars[i]
    return ''.join(chars)

def remove_diacritics(text):
    """Xoá dấu tiếng Việt."""
    nfkd = unicodedata.normalize('NFKD', text)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

PERTURBATIONS = {
    'Clean (gốc)':  lambda t: t,
    'Typo 10%':     lambda t: add_typo(t, 0.10),
    'Typo 20%':     lambda t: add_typo(t, 0.20),
    'Không dấu':    remove_diacritics,
}

robust_res = {}
models_to_test = [('KimCNN', ALL['KimCNN']['model'], True),
                  ('BiLSTM+Attention', ALL['BiLSTM+Attention']['model'], True)]

print('  Robustness Test')
print(f'  {"Điều kiện":<18}  {"KimCNN":>10}  {"BiLSTM+Attn":>13}')
print(f'  {"─"*46}')

for pert_name, pert_fn in PERTURBATIONS.items():
    row = []
    for mname, mmodel, use_v in models_to_test:
        perturbed = [pert_fn(t) for t in X_te]
        ds  = FeedbackDataset(perturbed, y_te, vocab=vocab)
        ld  = DataLoader(ds, batch_size=BATCH_SIZE)
        acc = evaluate_model(mmodel, ld)['accuracy']
        row.append(acc)
    robust_res[pert_name] = row
    print(f'  {pert_name:<18}  {row[0]:>10.4f}  {row[1]:>13.4f}')

# ── Biểu đồ robustness ─────────────────────────────────
conds  = list(robust_res.keys())
cnn_v  = [robust_res[c][0] for c in conds]
lstm_v = [robust_res[c][1] for c in conds]
x      = np.arange(len(conds)); w = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, cnn_v,  w, label='KimCNN',       color='#42A5F5', edgecolor='white')
ax.bar(x + w/2, lstm_v, w, label='BiLSTM+Attn',  color='#66BB6A', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(conds)
ax.set_ylim(0.4, 1.0); ax.set_ylabel('Accuracy')
ax.set_title('Robustness — Accuracy khi thêm noise', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('/content/outputs/robustness.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(' Saved: /content/outputs/robustness.png')


  Robustness Test
  Điều kiện               KimCNN    BiLSTM+Attn
  ──────────────────────────────────────────────
  Clean (gốc)             1.0000         1.0000
  Typo 10%                0.9489         0.8613
  Typo 20%                0.7786         0.6058
  Không dấu               0.3163         0.3698
 Saved: /content/outputs/robustness.png


## 📥 Phần 13 — Tải Kết Quả

In [27]:
import shutil
from google.colab import files

# Kiểm tra tất cả file
print(' /content/outputs/:')
for fname in sorted(os.listdir('/content/outputs')):
    kb = os.path.getsize(f'/content/outputs/{fname}') / 1024
    print(f'   {fname:<45} {kb:>7.1f} KB')

# Zip
shutil.make_archive('/content/nlp_outputs', 'zip', '/content/outputs')
zip_kb = os.path.getsize('/content/nlp_outputs.zip') / 1024
print(f'\n nlp_outputs.zip: {zip_kb:.1f} KB')
files.download('/content/nlp_outputs.zip')


 /content/outputs/:
   ablation.png                                     39.6 KB
   cm_bilstm.png                                    45.9 KB
   cm_cnn.png                                       44.5 KB
   cm_phobert.png                                   44.6 KB
   cm_rcnn.png                                      43.9 KB
   cm_transformer.png                               46.6 KB
   eda.png                                          75.4 KB
   history_bilstm.png                               54.2 KB
   history_cnn.png                                  59.0 KB
   history_phobert.png                              56.5 KB
   history_rcnn.png                                 53.9 KB
   history_transformer.png                          60.8 KB
   model_comparison.png                             48.6 KB
   results_table.csv                                 0.2 KB
   robustness.png                                   37.4 KB

 nlp_outputs.zip: 604.1 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>